# Data View

In [ ]:
import pandas as pd
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/blood_tests/blood_tests.parquet'
df_btest = read_parquet(pfile)
print(df_btest.shape)
df_btest.head()

In [ ]:
missing_analyzer = MissingValueAnalyzer(df_btest)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

# Imputation

In [ ]:
btest_metadata_col = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'hmo']
btest_metadata_col = [col for col in btest_metadata_col if col in df_btest.columns]
df_btest_metadata = df_btest[btest_metadata_col]
df_btest_features = df_btest.drop(columns=btest_metadata_col)

In [ ]:
# 删除缺失率超过50%的特征
threshold = 0.5
missing_percentage = df_btest_features.isnull().mean()
df_filtered = df_btest_features.loc[:, missing_percentage < threshold]
print(f"Number of features after filtering: {df_filtered.shape[1]}")
df_filtered.head()

In [ ]:
means = df_filtered.mean()
df_btest_imputed = df_filtered.fillna(means)

# Save

In [ ]:
df_btest_metadata.reset_index(drop=True, inplace=True)
df_btest_imputed.reset_index(drop=True, inplace=True)
df_btest = pd.concat([df_btest_metadata, df_btest_imputed], axis=1).dropna(axis=1)
pfile = 'pheno_data/202512/016-blood_tests_imputed.parquet'
df_btest.to_parquet(pfile, index=False)

In [ ]:
pfile = 'pheno_data/202512/016-blood_tests_imputed.parquet'
df_btest = pd.read_parquet(pfile)
print(df_btest.shape)
df_btest.head()